In [1]:
!pip install huggingface_hub

In [2]:
import os
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("kaggle_hf_key")
api = HfApi()
print(f"Logged in as: {api.whoami()['name']}")

Logged in as: equalNull


In [3]:
def find_global_columns(parquet_folder, con):
    cur_columns = set()
    for file in os.listdir(parquet_folder):
        if file.endswith(".parquet"):
            path = os.path.join(parquet_folder, file)
            # Read ONLY schema (no heavy data load)
            result = con.query(f"""
                DESCRIBE SELECT * FROM read_parquet('{path}')
            """).fetchall()
            cols = [row[0] for row in result]
            cur_columns.update(cols)
    return cur_columns

In [4]:
import duckdb, shutil
from huggingface_hub import snapshot_download

repo_id = f"equalNull/hosts100-labelled-optc"
all_columns = set()
working_path = "/kaggle/temp"
os.makedirs(working_path, exist_ok=True)
con = duckdb.connect(database=':memory:')
for day in range(16, 26):
    downloaded_path = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=working_path,
        allow_patterns=f"sep{day}/*",
        cache_dir=working_path,
        max_workers=4
    )

    cur_columns = find_global_columns(parquet_folder=f"{working_path}/sep{day}/", con=con)
    all_columns.update(cur_columns)
    print(f"Total {len(all_columns)} columns")
    shutil.rmtree(working_path)
    os.makedirs(working_path, exist_ok=True)

Fetching 93 files:   0%|          | 0/93 [00:00<?, ?it/s]

Total 62 columns


Fetching 93 files:   0%|          | 0/93 [00:00<?, ?it/s]

Total 62 columns


Fetching 100 files:   0%|          | 0/100 [00:00<?, ?it/s]

...

Total 62 columns


Fetching 100 files:   0%|          | 0/100 [00:00<?, ?it/s]

Total 62 columns


Fetching 100 files:   0%|          | 0/100 [00:00<?, ?it/s]

Total 62 columns


In [5]:
sorted_columns = sorted(list(all_columns))
print(sorted_columns)

['action', 'actorID', 'acuity_level', 'base_address', 'command_line', 'context_info', 'data', 'dest_ip', 'dest_port', 'direction', 'end_time', 'file_path', 'hostname', 'id', 'image_path', 'info_class', 'is_malicious', 'key', 'l4protocol', 'logon_id', 'module_path', 'name', 'new_path', 'object', 'objectID', 'parent_image_path', 'path', 'payload', 'pid', 'ppid', 'principal', 'privileges', 'requesting_domain', 'requesting_logon_id', 'requesting_user', 'service_type', 'sid', 'size', 'src_ip', 'src_pid', 'src_port', 'src_tid', 'stack_base', 'stack_limit', 'start_address', 'start_time', 'start_type', 'subprocess_tag', 'task_name', 'task_pid', 'task_process_uuid', 'tgt_pid', 'tgt_pid_uuid', 'tgt_tid', 'tid', 'timestamp', 'type', 'user', 'user_name', 'user_stack_base', 'user_stack_limit', 'value']


In [6]:
def encode_value(v, K=65536):
    if v is None:
        return 0

    try:
        x = float(v)
        return x
    except:
        return hash(v) % K